#### v2

In [1]:
import json
import numpy as np
from scipy.stats import binomtest

In [2]:
#func assemble_file_paths(...) -> file_paths
def assemble_file_paths(model_name, variant_names, prompt_type):
    file_paths = []
    for dataset_type in ["gold", "random"]:
        path_list = []
        for variant in variant_names:
            path_list.append(create_paths_str(model_name, variant, prompt_type, dataset_type))
        file_paths.append(path_list)
    return file_paths



#func create_paths_str(...) -> file_path
def create_paths_str(model_name, variant_name, prompt_type, dataset_type):
    return f"outputs/{model_name}/{variant_name}/responses_{prompt_type}_synthetic_dataset_{variant_name}_{dataset_type}.json"



model_name = "claude-3-opus-20240229"
variant_names = [
    "linda_variant_one_to",
    "linda_variant_one_because",
    "linda_variant_one_sothat",
    "linda_variant_three",
]
prompt_type = "baseline"
file_paths = assemble_file_paths(model_name, variant_names, prompt_type)
file_paths

[['outputs/claude-3-opus-20240229/linda_variant_one_to/responses_baseline_synthetic_dataset_linda_variant_one_to_gold.json',
  'outputs/claude-3-opus-20240229/linda_variant_one_because/responses_baseline_synthetic_dataset_linda_variant_one_because_gold.json',
  'outputs/claude-3-opus-20240229/linda_variant_one_sothat/responses_baseline_synthetic_dataset_linda_variant_one_sothat_gold.json',
  'outputs/claude-3-opus-20240229/linda_variant_three/responses_baseline_synthetic_dataset_linda_variant_three_gold.json'],
 ['outputs/claude-3-opus-20240229/linda_variant_one_to/responses_baseline_synthetic_dataset_linda_variant_one_to_random.json',
  'outputs/claude-3-opus-20240229/linda_variant_one_because/responses_baseline_synthetic_dataset_linda_variant_one_because_random.json',
  'outputs/claude-3-opus-20240229/linda_variant_one_sothat/responses_baseline_synthetic_dataset_linda_variant_one_sothat_random.json',
  'outputs/claude-3-opus-20240229/linda_variant_three/responses_baseline_synthetic_d

In [3]:
#func collect_grades (file_paths) -> grades
#   file_paths.type = list; file_paths.shape = (2, a); a = number of dataset variants
#   grades.type = np.array; grades.shape = (2, b); b = a * n; n = length of each grades list
def collect_grades(file_paths):
    grades = []
    for i in range(2):
        grades_list = []
        for j in range(len(file_paths[0])):
            grades_list.extend(extract_output_results(file_paths[i][j]))
        grades.append(grades_list)
    return np.array(grades)



#func extract_output_results (file_path) -> grades
def extract_output_results(file_path):
    # Note: not secured against edge cases (like if key "init_grades" is anything else than ["[Correct]"] or ["[Incorrect]"])
    with open(file_path, 'r') as f:
        data = json.load(f)
    length = data["stats"]["count_total"]
    grades = np.full(length, False)
    for i in range(length):
        grade = data[f"{i}"]["init_grades"][0]
        grades[i] = grade == "[Correct]"
    return grades



grades = collect_grades(file_paths)
grades.shape

(2, 400)

In [4]:
#func calc_test_statistics (grades) -> n12, n21, n_star, z0
#   grades.type = np.array; grades.shape = (2, b)

def calc_test_statistics (grades):
    grades_gold, grades_random = grades
    n12 = 0; n21 = 0
    for i in range(len(grades_gold)) :
        n12 += grades_gold[i] and not grades_random[i]
        n21 += not grades_gold[i] and grades_random[i]
    n_star = n12 + n21
    z0 = (n21 - n12) / np.sqrt(n_star)
    p_value = binomtest(n21, n_star, alternative='greater').pvalue
    return n12, n21, n_star, z0, p_value



calc_test_statistics(grades)

(16, 175, 191, 11.504836224149503, 2.7525638867682486e-35)

#### v1

In [5]:
import json
import numpy as np

In [6]:
file_1 = "outputs/claude-3-opus-20240229/linda_variant_one_to/responses_baseline_synthetic_dataset_linda_variant_one_to_gold.json"
file_2 = "outputs/claude-3-opus-20240229/linda_variant_one_to/responses_baseline_synthetic_dataset_linda_variant_one_to_random.json"

In [7]:
#func extract_output_results (file_1, file_2) -> grades_1, grades_2

def extract_output_results(file_path):
    # Note: not secured against edge cases (like if key "init_grades" is anything else than ["[Correct]"] or ["[Incorrect]"])
    with open(file_path, 'r') as f:
        data = json.load(f)
    length = data["stats"]["count_total"]
    grades = np.full(length, False)
    for i in range(length):
        grade = data[f"{i}"]["init_grades"][0]
        grades[i] = grade == "[Correct]"
    return grades

grades_1 = extract_output_results(file_1)
grades_2 = extract_output_results(file_2)

In [8]:
#func calc_test_statistics (grades_1, grades_2) -> n12, n21, n_star, z0

def calc_test_statistics (grades_1, grades_2):
    n12 = 0; n21 = 0
    for i in range(len(grades_1)) :
        n12 += grades_1[i] and not grades_2[i]
        n21 += not grades_1[i] and grades_2[i]
    n_star = n12 + n21
    z0 = (n21 - n12) / np.sqrt(n_star)
    p_value = binomtest(n21, n_star, alternative='greater').pvalue

    return n12, n21, n_star, z0, p_value

calc_test_statistics(grades_1, grades_2)

(1, 45, 46, 6.487446070815474, 6.679101716144942e-13)